# Phase 2: Dataset Cleaning & Standard RGB Preprocessing (No Green Injection)

This notebook is **identical** to `1_Dataset_Cleaning_and_Enhanced_Preprocessing.ipynb` EXCEPT:
- **NO green channel CLAHE injection** — images keep their natural RGB channels
- Only standard CLAHE on the L-channel (LAB color space) is applied for contrast enhancement

### Why?
The V1 Attention UNet (with standard RGB) and V2 (with green injection) produced very similar results.
We want to compare both preprocessing approaches fairly across all models to determine which works best.

### What This Notebook Does:
1. **Data Cleaning:** Removes corrupted images + all IDRiD images
2. **Standard Preprocessing:** Applies ONLY standard CLAHE (LAB L-channel) — no green manipulation
3. **Data Splitting:** Generates per-dataset + combined split CSVs
4. **Export:** Zips the output dataset ready for training

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import zipfile

In [ ]:
# ============================================================
# CONFIGURE YOUR PATHS HERE
# ============================================================

zip_path = "DR_dataset_segmentation_processed.zip"    # Your original dataset zip
extract_dir = "dataset"                                # Where to extract

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

BASE_INPUT_DIR = 'dataset'

# Output directory — DIFFERENT from the green-injection version
BASE_OUTPUT_DIR = 'dataset_stage1_v2_standard_rgb'
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_OUTPUT_DIR, 'images'), exist_ok=True)

print(f'Input:  {BASE_INPUT_DIR}')
print(f'Output: {BASE_OUTPUT_DIR}')

### 1. Define Corrupted Images (The Blacklist) 🚫

Same blacklist as the green-injection version — removes corrupted RITE images and ALL IDRiD images.

In [ ]:
blacklisted_images = [
    'RETINOMIX_RITE_train_image_52','RETINOMIX_RITE_train_image_51',
    'RETINOMIX_RITE_train_image_50','RETINOMIX_RITE_train_image_49',
    'IDRID_test_IDRiD_55','IDRID_test_IDRiD_56','IDRID_test_IDRiD_57',
    'IDRID_test_IDRiD_58','IDRID_test_IDRiD_59','IDRID_test_IDRiD_60',
    'IDRID_test_IDRiD_61','IDRID_test_IDRiD_63','IDRID_test_IDRiD_64',
    'IDRID_test_IDRiD_65','IDRID_test_IDRiD_66','IDRID_test_IDRiD_67',
    'IDRID_test_IDRiD_68','IDRID_test_IDRiD_69','IDRID_test_IDRiD_70',
    'IDRID_test_IDRiD_71','IDRID_test_IDRiD_72','IDRID_test_IDRiD_73',
    'IDRID_test_IDRiD_74','IDRID_test_IDRiD_75','IDRID_test_IDRiD_76',
    'IDRID_test_IDRiD_77','IDRID_test_IDRiD_78','IDRID_test_IDRiD_79',
    'IDRID_test_IDRiD_80','IDRID_test_IDRiD_81',
    'IDRID_train_IDRiD_01','IDRID_train_IDRiD_02','IDRID_train_IDRiD_03',
    'IDRID_train_IDRiD_04','IDRID_train_IDRiD_05','IDRID_train_IDRiD_06',
    'IDRID_train_IDRiD_07','IDRID_train_IDRiD_08','IDRID_train_IDRiD_09',
    'IDRID_train_IDRiD_10','IDRID_train_IDRiD_11','IDRID_train_IDRiD_12',
    'IDRID_train_IDRiD_13','IDRID_train_IDRiD_14','IDRID_train_IDRiD_15',
    'IDRID_train_IDRiD_16','IDRID_train_IDRiD_17','IDRID_train_IDRiD_18',
    'IDRID_train_IDRiD_19','IDRID_train_IDRiD_20','IDRID_train_IDRiD_21',
    'IDRID_train_IDRiD_22','IDRID_train_IDRiD_23','IDRID_train_IDRiD_24',
    'IDRID_train_IDRiD_25','IDRID_train_IDRiD_26','IDRID_train_IDRiD_27',
    'IDRID_train_IDRiD_28','IDRID_train_IDRiD_29','IDRID_train_IDRiD_30',
    'IDRID_train_IDRiD_31','IDRID_train_IDRiD_32','IDRID_train_IDRiD_33',
    'IDRID_train_IDRiD_34','IDRID_train_IDRiD_35','IDRID_train_IDRiD_36',
    'IDRID_train_IDRiD_37','IDRID_train_IDRiD_38','IDRID_train_IDRiD_39',
    'IDRID_train_IDRiD_40','IDRID_train_IDRiD_41','IDRID_train_IDRiD_42',
    'IDRID_train_IDRiD_43','IDRID_train_IDRiD_44','IDRID_train_IDRiD_45',
    'IDRID_train_IDRiD_46','IDRID_train_IDRiD_47','IDRID_train_IDRiD_48',
    'IDRID_train_IDRiD_49','IDRID_train_IDRiD_50','IDRID_train_IDRiD_51',
    'IDRID_train_IDRiD_52','IDRID_train_IDRiD_53','IDRID_train_IDRiD_54'
]

print(f'Loaded {len(blacklisted_images)} corrupted/excluded images.')

### 2. Standard Preprocessing (NO Green Injection) 🔬

**ONLY** standard CLAHE on the L-channel in LAB color space.  
The green channel is **NOT** replaced — images remain natural RGB.

In [ ]:
def apply_standard_clahe(image_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    """
    Standard CLAHE on L-channel (LAB color space).
    This is the SAME preprocessing used in V1.
    No green channel manipulation.
    """
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_ch, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_enhanced = clahe.apply(l_ch)
    enhanced_lab = cv2.merge([l_enhanced, a, b])
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)


def preprocess_fundus_standard(image_path):
    """
    Standard RGB preprocessing — NO green injection.
    Only applies standard CLAHE for contrast enhancement.
    """
    image = cv2.imread(image_path)
    if image is None:
        return None
    return apply_standard_clahe(image)


print('Standard preprocessing function ready (NO green injection)')

In [ ]:
# ============================================================
# Quick Visual Comparison: Standard vs Green Injection
# ============================================================

def apply_green_clahe(image_bgr, clip_limit=4.0, tile_grid=(4, 4), invert=True):
    """Green injection method (shown for comparison only, NOT used here)."""
    green = image_bgr[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    green_enhanced = clahe.apply(green)
    if invert:
        green_enhanced = 255 - green_enhanced
    return green_enhanced


def visualize_comparison(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Image not found: {image_path}')
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Standard (what THIS notebook produces)
    standard = apply_standard_clahe(img)
    standard_rgb = cv2.cvtColor(standard, cv2.COLOR_BGR2RGB)

    # Green injection (what the V2 notebook produces — shown for comparison)
    green_injected = apply_standard_clahe(img).copy()
    green_injected[:, :, 1] = apply_green_clahe(img, clip_limit=4.0, tile_grid=(4, 4), invert=False)
    green_injected_rgb = cv2.cvtColor(green_injected, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 3, 1)
    plt.title('Original', fontsize=13)
    plt.imshow(img_rgb)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title('Standard CLAHE (This Notebook) \u2705', fontsize=13, color='green')
    plt.imshow(standard_rgb)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title('Green Injection (V2 - NOT used here) \u274c', fontsize=13, color='red')
    plt.imshow(green_injected_rgb)
    plt.axis('off')

    plt.tight_layout()
    plt.show()


# Test on sample images
sample_images = [
    os.path.join(BASE_INPUT_DIR, 'images/MAPLES_train_20051201_37462_0400_PP.png'),
    os.path.join(BASE_INPUT_DIR, 'images/VESSEL_train_Image_08L_CHASE.png'),
    os.path.join(BASE_INPUT_DIR, 'images/VESSEL_train_12_dr_HRF.png'),
    os.path.join(BASE_INPUT_DIR, 'images/RETINOMIX_RITE_test_image_09.png'),
    os.path.join(BASE_INPUT_DIR, 'images/RETINOMIX_DRIVE_train_image_14.png'),
]

for img_path in sample_images:
    if os.path.exists(img_path):
        visualize_comparison(img_path)

### 3. Build & Clean the Dataset ⚒️

Loop through old CSVs, exclude bad images, apply **standard** preprocessing (no green), copy masks, create per-dataset splits.

In [ ]:
splits = ['train_split', 'val_split', 'test_split']
combined_dfs = {s: [] for s in splits}
maples_dfs   = {s: [] for s in splits}
drive_dfs    = {s: [] for s in splits}
chase_dfs    = {s: [] for s in splits}
rite_dfs     = {s: [] for s in splits}
hrf_dfs      = {s: [] for s in splits}

# Copy masks (masks don't need re-processing)
print('Copying masks...')
old_masks_dir = os.path.join(BASE_INPUT_DIR, 'lesion_masks')
new_masks_dir = os.path.join(BASE_OUTPUT_DIR, 'lesion_masks')
if not os.path.exists(new_masks_dir):
    shutil.copytree(old_masks_dir, new_masks_dir)
    print(f'  Lesion masks copied to {new_masks_dir}')

old_vessel_dir = os.path.join(BASE_INPUT_DIR, 'vessel_masks')
new_vessel_dir = os.path.join(BASE_OUTPUT_DIR, 'vessel_masks')
if not os.path.exists(new_vessel_dir) and os.path.exists(old_vessel_dir):
    shutil.copytree(old_vessel_dir, new_vessel_dir)
    print(f'  Vessel masks copied to {new_vessel_dir}')

# Process images
total_processed = 0
total_excluded = 0

for split in splits:
    old_csv_path = os.path.join(BASE_INPUT_DIR, f'{split}.csv')
    df = pd.read_csv(old_csv_path)

    processed_rows = []
    print(f'\nProcessing {split} split ({len(df)} images)...')
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_filename = row['img_id']

        # --- 1. FILTER: Remove blacklisted + IDRiD ---
        if img_filename in blacklisted_images or 'IDRiD' in img_filename:
            total_excluded += 1
            continue

        # --- 2. STANDARD PREPROCESSING (NO GREEN INJECTION) ---
        old_img_path = os.path.join(BASE_INPUT_DIR, row['img_path'])
        new_img_rel_path = os.path.join('images', img_filename)
        new_img_full_path = os.path.join(BASE_OUTPUT_DIR, new_img_rel_path)

        if not os.path.exists(new_img_full_path):
            enhanced_img = preprocess_fundus_standard(old_img_path)
            if enhanced_img is not None:
                cv2.imwrite(new_img_full_path, enhanced_img)
                total_processed += 1
            else:
                print(f'  WARNING: Could not read {old_img_path}')
                continue

        processed_rows.append(row)

    new_df = pd.DataFrame(processed_rows)
    combined_dfs[split] = new_df

    # Split by dataset source
    maples_dfs[split] = new_df[new_df['img_id'].str.contains('MAPLES|MESSIDOR', case=False, na=False)]
    drive_dfs[split]  = new_df[new_df['img_id'].str.contains('DRIVE', case=False, na=False)]
    chase_dfs[split]  = new_df[new_df['img_id'].str.contains('CHASE', case=False, na=False)]
    rite_dfs[split]   = new_df[new_df['img_id'].str.contains('RITE', case=False, na=False)]
    hrf_dfs[split]    = new_df[new_df['img_id'].str.contains('HRF', case=False, na=False)]

    # Save combined CSV
    new_df.to_csv(os.path.join(BASE_OUTPUT_DIR, f'combined_{split}.csv'), index=False)

print(f'\n{"="*50}')
print(f'Total processed: {total_processed} images')
print(f'Total excluded:  {total_excluded} images')
print(f'{"="*50}')

### 4. Create Source-Specific CSVs 📁

In [ ]:
dataset_dfs = {
    'maples': maples_dfs,
    'drive': drive_dfs,
    'chase': chase_dfs,
    'rite': rite_dfs,
    'hrf': hrf_dfs,
}

print('Per-dataset split CSVs:')
print(f'{"Dataset":<10} {"Train":>6} {"Val":>6} {"Test":>6}')
print('-' * 32)

for ds_name, ds_dfs in dataset_dfs.items():
    counts = []
    for split in splits:
        if len(ds_dfs[split]) > 0:
            ds_dfs[split].to_csv(os.path.join(BASE_OUTPUT_DIR, f'{ds_name}_{split}.csv'), index=False)
        counts.append(len(ds_dfs[split]))
    print(f'{ds_name:<10} {counts[0]:>6} {counts[1]:>6} {counts[2]:>6}')

# Combined totals
combined_counts = [len(combined_dfs[s]) for s in splits]
print('-' * 32)
print(f'{"COMBINED":<10} {combined_counts[0]:>6} {combined_counts[1]:>6} {combined_counts[2]:>6}')

print('\n\u2705 All CSVs created successfully!')

### 5. Zip & Download 📦

In [ ]:
!zip -r dataset_stage1_v2_standard_rgb.zip dataset_stage1_v2_standard_rgb/
print('\n\u2705 Done! Download dataset_stage1_v2_standard_rgb.zip and upload to Kaggle for training.')
print('This dataset uses STANDARD RGB preprocessing (no green channel injection).')

### 6. Summary: What Changed vs Green Injection Version

| Aspect | Green Injection (V2) | Standard RGB (This Notebook) |
|--------|---------------------|-----------------------------|
| **Preprocessing** | CLAHE + Green channel replaced | CLAHE only (LAB L-channel) |
| **Output dir** | `dataset_stage1_v2_enhanced` | `dataset_stage1_v2_standard_rgb` |
| **Data cleaning** | Same blacklist | Same blacklist |
| **IDRiD removed** | Yes | Yes |
| **Split structure** | Per-dataset + combined | Per-dataset + combined |
| **Mask handling** | Copied as-is | Copied as-is |

Both datasets have **identical splits** and **identical masks**. The ONLY difference is the image preprocessing.